## Delta Table API
The delta table allows one to interact with tables directly, via a reference to the actual table.  This eliminates the dataframe 'middle man'

- With the **DataFrame** API, there are only two modes for table interaction 'overwite' and 'append'.  Also, it does not allow individual record manipulation or CRUD actions into a single operation. In contrast the DetlaTable API is more robust
- Docs:  https://docs.delta.io/api/latest/python/spark/

In [0]:
"""
Import the DeltaTable class
"""

from delta.tables import DeltaTable
print(type(DeltaTable))

In [0]:
"""
first, read in as dataframe and display data
This is an instance of the 'dataframe' object
"""
df = spark.read.table('workspace.pyspark_learning.country_regions')
display(df)
print(type(df))

In [0]:
"""
Syntax for read in country_regions table not as a dataframe but as a delta table object
"""
dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_regions')
type(dt)

In [0]:
"""
Syntax for read in parquet file as a delta table
"""
dt2 = DeltaTable.forPath(spark, '/Volumes/workspace/pyspark_learning/raw_files/pyspark/countries_dataset/detaLake/')
type(dt2)

Record Munipulation - CRUD
- delta table api interacts with the delta tables in the catalog directly via the delta object in the cell.  It needs no dataframe as a 'middle-man'.
- Whereas the delta table API interacts with individual table records, if using a dataframe, one must edit the dataframe then 'overwrite' a table to remove or update records

### Using the Delta API to Delete Records

In [0]:
"""
Delete record
NOTE:  This does not delete a record locally in the current session like in a dataframe, but 'dt' is a reference to the table itself.  What one executes on the delta table object directly affects the table
"""
dt.delete("name = 'America'")

In [0]:
"""
After delete using the dt object
Display 'df', and the record with America is gone.  Compare to cell #2 above
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()


### Using the Delta API to Update Records

In [0]:
"""
Import functions
"""

from pyspark.sql.functions import lit, upper


In [0]:
"""
for records in table column, 'name' where the value is 'Asia', set the id to 100 and name to UPPERCASE
    step 1. identify the condition (the 'where' clause)
    step 2. set the column values
Note: set uses a dictionary object {'column name':  'value'}
"""
dt.update(
    condition = "name = 'Asia'",
    set = {
        "id": lit(100),
        "name": upper("name")
    }
)

In [0]:
"""
verify update via dataframe read table
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()


In [0]:
"""
To update ALL records, set 'condition = None', meaning greedy update
Below I am setting all name values to uppercase
"""
dt.update(
    condition = None,
    set = {
        "name": upper("name")
    }
)

In [0]:
"""
verify update via dataframe read table
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

### Merge Records
'merge' performs an update where exists and insert where does not, 'upsert'

In [0]:
"""
import all classes
"""
from delta.tables import *

In [0]:
"""
import table and create delta object
"""
dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_regions')
type(dt)

In [0]:
"""
a delta object CANNOT be directly displayed so it needs to be converted to a dataframe first for display purposes
NOTE:  chained methods
"""
dt.toDF().display()

In [0]:
"""
delete the all records where id = 100 using the dt object
"""
dt.delete("id = 100")

In [0]:
"""
delete the all records where id = 20 using the dt object
"""

dt.delete("id = 20")

In [0]:
"""
verify the table has been updated
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

In [0]:
"""
PREPARE for UPSERT
read original regions csv data into dataframe
ddl schema is used here for simplicity
"""
schema_1 = "id int, name string"
regions_df = spark.read.csv('/Volumes/workspace/pyspark_learning/raw_files/pyspark/countries_dataset/csv_data/country_regions/country_regions.csv', header=True, schema=schema_1)
display(regions_df)
regions_df.printSchema()

In [0]:
"""
UPSERT
use regions_df to upsert into the country_regions table, inserting and updating records
- When ID's match UPDATE from dataframe to table
- When ID's do not match INSERT from dataframe into table

Target is on the left, source is on the right of each statement
dt is the table, aliased to 'target'
regions_df is is the dataframe, aliased to 'source'

We are updating and inserting from source INTO target

Expected Results:
names are UPDATED back to lowercase
deleted records re-inserted into table

NOTE:  set uses dictionary object
"""

dt.alias("target").\
    merge(
        regions_df.alias("source"),
        "target.id = source.id"
    ).\
    whenMatchedUpdate(
        set = {
            "target.name": "source.name",
        }
    ).\
    whenNotMatchedInsert(
        values = {
            "target.id" : "source.id",
            "target.name": "source.name"
        }
    ).execute()
            

In [0]:
"""
Verify table was updated via dataframe object
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

## DELTA Transaction log

In [0]:
"""
Create data for transaction log
NOTE:  the write method has no '.delta' method to chain; therefore CANNOT do:  df.write.delta.  Must use 'format' method with chained save method.

data is a list of dictionaries
"""
data = [
  {"id": 1, "name": "Alice", "score": 85},
  {"id": 2, "name": "Bob", "score": 90},
  {"id": 3, "name": "Charlie", "score": 70},
  {"id": 4, "name": "Diana", "score": 92}, 
  {"id": 5, "name": "Ethan", "score": 88},
  {"id": 6, "name": "Fiona", "score": 81},
  {"id": 7, "name": "George", "score": 74},
  {"id": 8, "name": "Hannah", "score": 95},
  {"id": 9, "name": "Ian", "score": 69},
  {"id": 10, "name": "Jasmine", "score": 87},

]

# create data and save to volume ('file system')
df = spark.createDataFrame(data)
df.write.format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data', mode='append')



In [0]:
"""
add more students to enhance log
"""

data2 = [
  {"id": 11, "name": "Kevin", "score": 83},
  {"id": 12, "name": "Lena", "score": 91},
  {"id": 13, "name": "Marcus", "score": 77},
  {"id": 14, "name": "Nina", "score": 89}, 
  {"id": 15, "name": "Oscar", "score": 72},
  {"id": 16, "name": "Priya", "score": 94},
  {"id": 17, "name": "Quinn", "score": 79},
  {"id": 18, "name": "Ravi", "score": 88},
  {"id": 19, "name": "Sophie", "score": 86},
  {"id": 20, "name": "Tom", "score": 80},

]

df2 = spark.createDataFrame(data2)
df2.write.format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data', mode='append')

In [0]:
"""
Delete all scores less than 80
"""
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, '/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data')
dt.delete("score < 80")

In [0]:
"""
create more logs
"""

for i in range(50):
    spark.createDataFrame(data2).write.mode('append').format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data')

In [0]:
"""
list volume contents
passing 'dbutils.fs.ls' to the display is the equivalent of pprint
"""
display(dbutils.fs.ls('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data'))

### History and Timetravel 

By default, Delta retains 30 days of history only.  But is configurable

In [0]:
"""
Get table history via 'DESCRIBE'
Can provide a table name or a path to delta files in the volume

Shows all commits for this table, 0 - 53
NOTE:  Take a look at each column, its very detailed
"""
spark.sql("""
          DESCRIBE HISTORY delta.`/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data`
          """).display()

In [0]:
"""
to get table history, supply catalog.schema.table.
NOTE: no detla or backticks
"""

spark.sql("""
          DESCRIBE HISTORY workspace.pyspark_learning.country_regions
          """).display()

Looking at specific commits from history

In [0]:
"""
'versionAsOf'
One can use this history to look at any of the commits
NOTE: Look at the order of the command issued.  Must use:  read.option(<options>), then table last
"""
spark.read.option("versionAsOf", 9).table('workspace.pyspark_learning.country_regions').display()

In [0]:
"""
Delta file example, similar syntax, but 'format' and 'load' methods used
"""
spark.read.option('versionAsOf', 2).format('delta').load("/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data").display()

In [0]:
"""
Use timestamp instead of version number to get prior commits, see below
"""
spark.read.option('timestampAsOf', "2026-03-02T22:17:02.000+00:00").format('delta').load("/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data").display()

RESTORE from history

In [0]:
"""
Restore to prior timestamp via 'RESTORE' command.
Below restoring to version #0 of #53 versions
"""

spark.sql("""
          RESTORE delta.`/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data`
          TO VERSION AS OF 0
          """)

In [0]:
"""
Get table history via 'DESCRIBE'
We get another, additional commit, #54

Shows all commits for this table, #0 - #54
NOTE: the 'RESTORE' in the 'operation' column
"""
spark.sql("""
          DESCRIBE HISTORY delta.`/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data`
          """).display()

### SLOWLY CHANGEING DIMENSIONS (SCD)

Schema Evolution

This is the ability to change a table's scheam without dropping and recreating the table

In [0]:
"""
Verify current sub regions table data
"""

spark.read.table('workspace.pyspark_learning.country_sub_regions').display()

In [0]:

"""
create test data, data_frame with 3 columns:
id, name, new_col
"""
from pyspark.sql.functions import *
from pyspark.sql.types import *

new_df = spark.createDataFrame([
    {'id': 100, "name": "New Sub Region", "new_col": "Some New Value"}
])

new_df = new_df.select(col("id").cast(IntegerType()), col("name").cast(StringType()), col("new_col").cast(StringType()))

new_df.display()

In [0]:
"""
Attempt to append the the new 3 column dataframe to the 2 column sub_regions table
This should throw a 'schema mismatch' error
"""

new_df.write.saveAsTable('workspace.pyspark_learning.country_sub_regions', mode='append')

In [0]:
"""
rerun with the 'mergeSchema' option
verify results
"""

new_df.write.saveAsTable('workspace.pyspark_learning.country_sub_regions', mode='append', mergeSchema=True)
spark.read.table('workspace.pyspark_learning.country_sub_regions').display()

In [0]:
"""
RESTORE table back using history,
First, get the history to determine versions
"""
spark.sql("""
          DESCRIBE HISTORY workspace.pyspark_learning.country_sub_regions
          """).display()


In [0]:
"""
RESTORE to version #1
"""

spark.sql("""
          RESTORE workspace.pyspark_learning.country_sub_regions
          TO VERSION AS OF 5
          """)

In [0]:
"""
If versions are too old to get version, can reimport the table, or delete the extra column
"""
spark.sql("""ALTER TABLE workspace.pyspark_learning.country_sub_regions SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')""")
spark.sql("""ALTER TABLE workspace.pyspark_learning.country_sub_regions DROP COLUMN new_col""")

In [0]:
"""
delete the record which was added with the new column
"""
new_dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_sub_regions')
new_dt.delete("name = 'New Sub Region'")

In [0]:
spark.read.table('workspace.pyspark_learning.country_sub_regions').display()


### Slowly Changing Dimesions

Dimension data changes over time.  This data which is descriptive of 'facts' information.

- Type 1:  Overwrite attribute values with new ones.  No history is retained
- Type 2:  Data history is retained.  There 'EFFECTIVE_DATE' and 'END_DATE' columns for the descriptive attribute data.  If there is no 'END_DATE' for the record (it's NULL), that means the value is active or currently applied to the data.  If an attribute changes, its "EFFECTIVE_DATE' is the same value as the 'END_DATE' of the prior value, and this new value's 'END_DATE' is NULL.  There will be a new row added every time an attribute changes.

In [0]:
"""
CREATE SCD TYPE 2 TABLE
Uses country regions as example
"""

spark.sql(
    """
    CREATE TABLE workspace.pyspark_learning.country_regions_scd_2
    (
        id INT,
        name STRING,
        effective_date TIMESTAMP,
        end_date TIMESTAMP
    )
    """
)


In [0]:
"""
verify empty table
"""
spark.read.table('workspace.pyspark_learning.country_regions_scd_2').display()

In [0]:
"""
load seed data with ddl type schema
"""
schema1 = ("id int, name string")
df_changes = spark.read.csv(
    '/Volumes/workspace/pyspark_learning/raw_files/pyspark/countries_dataset/csv_data/country_regions/', 
    header=True, 
    schema=schema1
)
df_changes.display()

In [0]:
"""
MERGE (update) data in the table using the delta table api
STEP 1: Update current row if condition is met
the code below merges the df_changes data into the table
"""

from delta.tables import DeltaTable
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import TimestampType, IntegerType, StringType

dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_regions_scd_2')

dt.alias("target") \
.merge(
    source = df_changes.alias("source"),
    condition = "target.id = source.id AND target.end_date IS NULL"
) \
.whenMatchedUpdate(
    set = {"target.end_date": current_timestamp()}
) \
.execute()



In [0]:
"""
STEP 2 Insert New records via APPEND using the dataframe API
Note:  Nothing actually gets saved to 'df_changes' variable.  It writes to the table and the variable contents are then 'NONE'

effective_date", current_timestamp() indictates the record is current as there is no end_date 'lit(None)'

"""
df_changes = df_changes.\
    withColumn("effective_date", current_timestamp()).\
    withColumn("end_date", lit(None).cast(TimestampType())).\
    select(df_changes.id.cast(IntegerType()), df_changes.name.cast(StringType()), "effective_date", "end_date").\
    write.mode('append').saveAsTable('workspace.pyspark_learning.country_regions_scd_2')

In [0]:
"""
Verify
Schema changed in table
"""
spark.read.table('workspace.pyspark_learning.country_regions_scd_2').display()

In [0]:
"""
create new data
We will update the America Record to United States

RERUN MERGE STEPS 1 and 2 AFTER creating this dataframe
"""
data = [
    {
        'id': 10,
        'name': 'United States'
    }
]

df_changes = spark.createDataFrame(data)
df_changes.display()

In [0]:
"""
Verify
You can see the id = 10, America record exists but is no longer active as it has an 'end_date' entry
Id 10, is now the active United States record
"""
spark.read.table('workspace.pyspark_learning.country_regions_scd_2').display()

In [0]:
"""
create new data
We will update the America Record to United States

RERUN MERGE STEPS 1 and 2 AFTER creating this dataframe
"""
data = [
    {
        'id': 10,
        'name': 'USA'
    }
]

df_changes = spark.createDataFrame(data)
df_changes.display()

In [0]:
"""
Verify
You can see the id = 10, United States record exists but is no longer active as it has an 'end_date' entry
Id 10, is now the active USA record
"""
spark.read.table('workspace.pyspark_learning.country_regions_scd_2').display()

In [0]:
"""
To see active records on SCD table, filter where end_date is null
"""
spark.sql('SELECT * FROM workspace.pyspark_learning.country_regions_scd_2 WHERE end_date IS NULL').display()